## Ship Presence Detection Pipeline with CNN

This notebook implements a binary image classifier to detect whether an image contains a ship or is empty (no ship present). The pipeline includes dataset preparation, TensorFlow dataset creation, CNN model construction, training, evaluation, and checkpointing.

## Dataset Preparation

In [ ]:
import numpy as np

from utils import load_masks

We load the masks into `df` and replace empty strings with `NaN` values. 
This standardizes missing masks for images with no ships.
Then we aggregate masks per image to create a binary label: 1 if a ship is present, 0 if empty.

In [ ]:
df = load_masks()
df['EncodedPixels'] = df['EncodedPixels'].replace('', np.nan)

df = df.groupby('ImageId')['EncodedPixels'] \
       .apply(lambda x: 1 if x.notna().any() else 0) \
       .reset_index()

df = df.rename(columns={'EncodedPixels': 'label'})

In [ ]:
df.head(5)

In [ ]:

from sklearn.model_selection import train_test_split

Split the dataset into train, validation, and test sets. Stratify by label to maintain class balance.

In [ ]:
train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

## TensorFlow Dataset Creation

In [ ]:
import os
import pandas as pd
import tensorflow as tf

from typing import Tuple
from constants import IMAGE_PATH

Define functions to load images, resize to 224x224, normalize, and return with labels.
Use `tf.py_function` to wrap for graph execution.

In [ ]:
def load_image(img_id: tf.Tensor, label: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    """
    Load an image from disk, decode, resize, normalize, and return it with its label.

    :param img_id: Tensor containing the image file name.
    :param label: Tensor containing the label corresponding to the image.
    :returns: Tuple of (processed image tensor, label tensor) both as tf.float32.
    """
    img_id_str = img_id.numpy().decode("utf-8")
    img_path = os.path.join(IMAGE_PATH, img_id_str)
    img = tf.io.read_file(str(img_path))
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = img / 255.0
    return tf.cast(img, tf.float32), tf.cast(label, tf.float32)
    
def tf_load_image(img_id: tf.Tensor, label: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor]:
    """
    TensorFlow wrapper for load_image using tf.py_function to allow graph execution.

    :param img_id: Tensor containing the image file name.
    :param label: Tensor containing the label corresponding to the image.
    :returns: Tuple of (processed image tensor, label tensor) both as tf.float32 with fixed shapes.
    """
    img, lbl = tf.py_function(load_image, [img_id, label], [tf.float32, tf.float32])
    img.set_shape([224, 224, 3])
    lbl.set_shape([])
    return img, lbl

def make_dataset(df: pd.DataFrame, batch_size: int = 32, shuffle: bool = True) -> tf.data.Dataset:
    """
    Create a TensorFlow dataset from a DataFrame of image paths and labels.

    :param df: Pandas DataFrame containing 'ImageId' and 'label' columns.
    :param batch_size: Number of samples per batch.
    :param shuffle: Boolean flag to shuffle the dataset.
    :returns: tf.data.Dataset ready for training or evaluation.
    """
    dataset = tf.data.Dataset.from_tensor_slices((df['ImageId'].values, df['label'].values))
    dataset = dataset.map(tf_load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

In [ ]:
train_ds = make_dataset(train_df)
val_ds   = make_dataset(val_df, shuffle=False)
test_ds  = make_dataset(test_df, shuffle=False)

## CNN Model for Ship Detection

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.metrics import AUC, Precision, Recall

The CNN learns to distinguish images with ships from empty ones by extracting relevant features and summarizing them for classification. The sigmoid output gives the probability of a ship being present. Binary crossentropy trains the model to predict accurate probabilities, while accuracy, AUC, precision, and recall measure different aspects of its performance.

In [ ]:
def make_empty_detector(input_shape: Tuple[int, int, int] = (224, 224, 3)) -> models.Sequential:
    """
    Create and compile a simple CNN model for binary classification.

    :param input_shape: Shape of the input images, e.g., (height, width, channels).
    :returns: Compiled Keras Sequential model ready for training.
    """
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv2D(16, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(32, 3, activation='relu', padding='same'),
        layers.MaxPooling2D(),
        layers.Conv2D(64, 3, activation='relu', padding='same'),
        layers.GlobalAveragePooling2D(),
        layers.Dense(32, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy', AUC(name='auc'), Precision(name='precision'), Recall(name='recall')]
    )
    return model

model = make_empty_detector()
model.summary()

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

## Training Setup

Compute class weights to balance ship vs empty images.
Setup callbacks: checkpointing best model and early stopping.

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.array([0,1]),
    y=train_df['label'].values
)
class_weight_dict = dict(enumerate(class_weights))

CHECKPOINT_PATH = "./checkpoints/best_cnn_model.h5"

checkpoint_cb = ModelCheckpoint(
    filepath=CHECKPOINT_PATH,
    monitor="val_loss",
    save_best_only=True,
    save_weights_only=False,
    mode="min",
    verbose=1
)

earlystop_cb = EarlyStopping(
    monitor="val_loss",
    patience=2,
    min_delta=0.01, 
    restore_best_weights=True,
    mode="min",
    verbose=1
)

In [ ]:
from tensorflow.keras.models import load_model, Model, Sequential
from tensorflow.keras.callbacks import Callback
from tensorflow.data import Dataset
from typing import Dict, Union

## Model Training / Loading
If a checkpoint exists, load the model. Otherwise, train a new model for 10 epochs using class weights.

In [ ]:
def get_model(
    train_ds: Dataset,
    val_ds: Dataset,
    class_weight_dict: Dict[int, float],
    checkpoint_cb: Callback,
    earlystop_cb: Callback
) -> Model:
    """
    Load an existing model from checkpoint if available; otherwise, create, train, and return the best model.

    :param train_ds: Training dataset (tf.data.Dataset).
    :param val_ds: Validation dataset (tf.data.Dataset).
    :param class_weight_dict: Dictionary mapping class indices to weights for handling class imbalance.
    :param checkpoint_cb: Keras Callback for model checkpointing.
    :param earlystop_cb: Keras Callback for early stopping during training.
    :returns: Trained Keras Model loaded from the best checkpoint.
    """
    if os.path.exists(CHECKPOINT_PATH):
        print(f"Loading existing model from {CHECKPOINT_PATH}")
        return load_model(CHECKPOINT_PATH)

    model = make_empty_detector()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=10,
        class_weight=class_weight_dict,
        callbacks=[checkpoint_cb, earlystop_cb]
    )
    best_model = load_model(CHECKPOINT_PATH)
    return best_model

In [ ]:
model = get_model(train_ds, val_ds, class_weight_dict, checkpoint_cb, earlystop_cb)

## Evaluation

Evaluate the model on the test set. Metrics include accuracy, AUC, precision, and recall.

In [ ]:
test_loss, test_acc = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.4f}")